# MLP Bottleneck Analysis

How information is compressed into the MLP hidden layer and expanded
back: compression ratio, utilization, reconstruction, selectivity.

## Why This Matters

MLP bottleneck provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_bottleneck_analysis import (
    mlp_compression_ratio, mlp_hidden_utilization,
    mlp_input_reconstruction, mlp_expansion_selectivity,
    mlp_bottleneck_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Compression Ratio

Effective rank of input vs hidden representations.

In [ ]:
for layer in range(4):
    result = mlp_compression_ratio(model, tokens, layer=layer)
    print(f"  Layer {layer}: input_rank={result['input_effective_rank']:.2f}, "
          f"hidden_rank={result['hidden_effective_rank']:.2f}, "
          f"ratio={result['compression_ratio']:.3f}")

## Hidden Utilization

What fraction of MLP neurons are actively used?

In [ ]:
for layer in range(4):
    result = mlp_hidden_utilization(model, tokens, layer=layer)
    bar = '█' * int(result['active_fraction'] * 40)
    print(f"  Layer {layer}: {result['n_active']}/{result['n_total']} active "
          f"({result['active_fraction']:.1%}) {bar}")

## Input Reconstruction

Does the MLP output point in the same direction as its input?

In [ ]:
for layer in range(4):
    result = mlp_input_reconstruction(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_cos={result['mean_reconstruction']:.4f}")

## Expansion Selectivity

In [ ]:
for layer in range(4):
    result = mlp_expansion_selectivity(model, tokens, layer=layer, top_k=3)
    most = [(n, f'{s:.3f}') for n, s in result['most_selective']]
    print(f"  Layer {layer}: mean_sel={result['mean_selectivity']:.4f}, "
          f"most_selective={most}")

## Bottleneck Summary

In [ ]:
result = mlp_bottleneck_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: comp={p['compression_ratio']:.3f}, "
          f"active={p['active_fraction']:.1%}")